# Week 5 — Urdu OCR: Gradio App + Hugging Face Spaces Deployment
### SI-26 | Code Saviours Summer Internship 2026
**Project:** Urdu OCR Tool | **Week:** 5 of 5 (Final Week) | **Submitted by:** Qandeel Asim

---

### What this notebook does
This is the final step of the project: it loads the model you fine-tuned in Week 4,
wraps it in a live, testable web app (Gradio), and packages everything needed to make
that app permanently public (Hugging Face Spaces).

**No training happens here.** Every cell below either *loads* something you already
built, or *uses* it. If a cell errors, the fix is almost always in Week 4 (the saved
model files), not in this notebook.

### How to run this notebook
Run every cell **top to bottom, in order, once**. Do not skip Section 2 (Drive mount),
and do not stop/interrupt Section 8 (the Gradio launch cell) while you're actively
testing the app in its browser tab — interrupting it kills the live server mid-request,
which is the single most common cause of "nothing happens after I click Submit."

### Sections
| # | Section | Purpose |
|---|---------|---------|
| 0 | Install Gradio | one-time setup |
| 1 | GPU check | confirms Colab gave you a GPU |
| 2 | Mount Drive & locate model | finds your Week 4 output automatically |
| 3 | Rebuild the Urdu tokenizer | turns model IDs back into Urdu text |
| 4 | Load the model | loads your fine-tuned weights |
| 5 | Inference function | the core `predict(image) -> text` logic |
| 6 | Sanity check | proves the model works *before* touching Gradio |
| 7 | Build sample gallery | a few real test images for one-click testing |
| 8 | Launch the Gradio app | the live, shareable demo |
| 9 | Package for Hugging Face Spaces | app.py + requirements.txt + model, zipped |
| 10 | Generate README draft | all 8 required sections, pre-filled |
| 11 | Submission checklist | what to hand in and by when |

## Section 0 — Install Gradio

In [ ]:
!pip install -q gradio
print("Gradio installed.")

Gradio installed.


## Section 1 — GPU Check

Inference works on CPU too, just slower per image (a few seconds instead of under one).
A GPU is nice to have here but not required, unlike in Week 4's training.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — inference will still work, just a bit slower per image.")

CUDA available: True
GPU: Tesla T4


## Section 2 — Mount Drive and Locate the Fine-Tuned Model

Two things happen here:
1. **Find the model.** Rather than hardcoding a path (which breaks the moment your
   Drive structure changes even slightly), this walks your entire Drive looking for a
   folder named `trocr-urdu-finetuned` that actually contains `urdu_char_vocab.pkl`
   — the one saved at the very end of your Week 4 notebook.
2. **Define `resolve_image_path()`.** Week 3 and Week 4 both hit the same recurring
   issue: image paths inside `labels.csv` don't always match exactly where the files
   ended up on Drive (extra nested folders from how Drive Sync works). This helper
   tries the path as-is first, and falls back to searching all of Drive by filename.
   We define it once here and reuse it in Sections 6 and 7 — no more "couldn't resolve
   path" surprises later in the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

MODEL_FOLDER_NAME = "trocr-urdu-finetuned"

found_model_dir = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if os.path.basename(root) == MODEL_FOLDER_NAME and "urdu_char_vocab.pkl" in files:
        found_model_dir = root
        break

if found_model_dir is None:
    raise FileNotFoundError(
        f"Could not find a '{MODEL_FOLDER_NAME}' folder (with urdu_char_vocab.pkl inside) "
        "anywhere in My Drive. Make sure the final save cell of the Week 4 notebook was "
        "run and has fully synced to Drive before running this notebook."
    )

MODEL_DIR = found_model_dir
PROJECT_DIR = os.path.dirname(MODEL_DIR)
print("Found fine-tuned model at:", MODEL_DIR)
print("Contents:", os.listdir(MODEL_DIR))


def resolve_image_path(image_path, project_dir):
    """Resolve an image_path from labels.csv to a real file on Drive, even if the
    folder structure shifted slightly. Tries the path as given first, then falls
    back to a filename search across all of Drive."""
    candidate = os.path.join(project_dir, image_path)
    if os.path.exists(candidate):
        return candidate

    filename = os.path.basename(image_path)
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if filename in files:
            return os.path.join(root, filename)
    return None

Mounted at /content/drive
Found fine-tuned model at: /content/drive/MyDrive/Classroom/Machine Learning | SI-26/trocr-urdu-finetuned
Contents: ['processor_config.json', 'tokenizer_config.json', 'tokenizer.json', 'preprocessor_config.json', 'generation_config.json', 'urdu_char_vocab.pkl', 'config.json', 'model.safetensors']


## Section 3 — Rebuild the Urdu Character Tokenizer (Decode-Only)

Week 4 used a **custom** `UrduCharTokenizer` class — not a Hugging Face tokenizer — so
it can't be loaded with `.from_pretrained()`. Only its vocabulary dictionary was saved
(`urdu_char_vocab.pkl`). For inference we only ever need to go **from model output IDs
back to Urdu text** (decoding), never the other direction, so this rebuilds just that
half of the original class.

In [ ]:
import pickle

class UrduCharTokenizerDecode:
    """Decode-only counterpart to the Week 4 UrduCharTokenizer, rebuilt from the saved vocab."""

    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {i: tok for tok, i in vocab.items()}
        self.pad_token_id = vocab["<pad>"]
        self.bos_token_id = vocab["<bos>"]
        self.eos_token_id = vocab["<eos>"]
        self.unk_token_id = vocab["<unk>"]

    def decode(self, ids, skip_special_tokens=True):
        out = []
        for i in ids:
            i = int(i)
            if skip_special_tokens and i in (self.pad_token_id, self.bos_token_id, self.eos_token_id):
                continue
            out.append(self.inv_vocab.get(i, ""))
        return "".join(out)

    def batch_decode(self, ids_batch, skip_special_tokens=True):
        return [self.decode(ids, skip_special_tokens=skip_special_tokens) for ids in ids_batch]


with open(os.path.join(MODEL_DIR, "urdu_char_vocab.pkl"), "rb") as f:
    urdu_vocab = pickle.load(f)

tokenizer = UrduCharTokenizerDecode(urdu_vocab)
print("Urdu character vocab size:", len(tokenizer.vocab))
assert len(tokenizer.vocab) > 4, "Vocab looks too small — check urdu_char_vocab.pkl was saved correctly in Week 4."
print("Tokenizer ready.")

Urdu character vocab size: 54
Tokenizer ready.


## Section 4 — Load the Fine-Tuned Model

Loads the exact weights you trained and evaluated in Week 4 — nothing is retrained or
re-initialized here.

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR)
image_processor = ViTImageProcessor.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()  # inference mode: disables dropout, etc. — always do this before predicting

print("Model loaded on:", device)
print("Model ready for inference.")

Loading weights:   0%|          | 0/278 [00:00<?, ?it/s]

Model loaded on: cuda
Model ready for inference.


## Section 5 — The Inference Function

This is the single function everything else in this notebook (sanity check, gallery,
Gradio app) calls. Keeping it in one place means the sanity check and the live app are
*always* guaranteed to behave identically — there's no separate "app version" of the
logic that could drift out of sync.

In [ ]:
from PIL import Image

@torch.no_grad()
def predict(image):
    """
    Extract Urdu text from an image using the fine-tuned TrOCR model.

    Parameters
    ----------
    image : PIL.Image.Image
        Any image containing Urdu text (photo, scan, screenshot).

    Returns
    -------
    str
        The predicted Urdu text, or "" if no image was given.
    """
    if image is None:
        return ""

    image = image.convert("RGB")
    pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return text

print("predict() is defined and ready.")

predict() is defined and ready.


## Section 6 — Sanity Check on a Real Test Image

**This is the most important cell in the notebook to check carefully.** If this cell
prints a reasonable Urdu sentence that's close to the ground truth, your model and
`predict()` function are both working correctly — any issue you see later in Gradio is
a *Gradio/UI* issue, not a model issue. If this cell fails or prints garbage, the
problem is in the model/tokenizer, and fixing Gradio won't help.

In [ ]:
import pandas as pd

labels_path = os.path.join(PROJECT_DIR, "labels.csv")

if not os.path.exists(labels_path):
    print(f"labels.csv not found at {labels_path} — skipping sanity check.")
    print("This does NOT affect the Gradio app below; it only skips this one check.")
else:
    df = pd.read_csv(labels_path)
    image_col = "image_path" if "image_path" in df.columns else df.columns[0]

    checked, matched = 0, 0
    for idx in range(min(3, len(df))):
        row = df.iloc[idx]
        resolved_path = resolve_image_path(row[image_col], PROJECT_DIR)
        checked += 1

        if resolved_path is None:
            print(f"[{idx}] Could not locate image on Drive: {row[image_col]} — skipped.")
            continue

        sample_image = Image.open(resolved_path)
        prediction = predict(sample_image)
        matched += 1

        print(f"--- Sample {idx} ---")
        print("Ground truth:", row["text"])
        print("Model output:", prediction)
        print()

    if matched == 0:
        print("No sample images could be located — this is a Drive path issue only, "
              "not a model issue. The Gradio app below is unaffected since it uses "
              "live-uploaded images, not labels.csv.")
    else:
        print(f"Sanity check complete: {matched}/{checked} samples predicted successfully.")
        print("If the outputs above look close to the ground truth, your model is working correctly.")

--- Sample 0 ---
Ground truth: برنارڈشا نے دو باتیں لکھی ہین جن سے اس خیال کو تقویت پہنچتی ہے کہ
Model output: برنارڈشا نے دو باتیں لکھی ہین جن سے اس خیال کو تقویت پر ہن ہو

--- Sample 1 ---
Ground truth: قرآن مجید مسلمانوں کی مقدس کتاب ہے
Model output: قرآن مجید مسلمانوں کی مقدس کتاب ہے

--- Sample 2 ---
Ground truth: ہمت مرداں مدد خدا
Model output: ہمت مرداں مدد خدا

Sanity check complete: 3/3 samples predicted successfully.
If the outputs above look close to the ground truth, your model is working correctly.


## Section 7 — Build a Sample Gallery for One-Click Testing

To avoid any confusion about "did I upload correctly / did I click Submit," this
section collects a handful of real images from your dataset so the Gradio app (next
section) can offer them as ready-made examples — one click, no upload needed, and
you can immediately see it's working end-to-end.

In [ ]:
example_image_paths = []

if os.path.exists(labels_path):
    df = pd.read_csv(labels_path)
    image_col = "image_path" if "image_path" in df.columns else df.columns[0]

    for idx in range(len(df)):
        if len(example_image_paths) >= 4:
            break
        row = df.iloc[idx]
        resolved_path = resolve_image_path(row[image_col], PROJECT_DIR)
        if resolved_path is not None:
            example_image_paths.append(resolved_path)

print(f"Found {len(example_image_paths)} example image(s) for the Gradio gallery:")
for p in example_image_paths:
    print(" -", p)

if not example_image_paths:
    print("No example images found — the app below will still work fine with manual "
          "uploads, it just won't show one-click examples.")

Found 4 example image(s) for the Gradio gallery:
 - /content/drive/MyDrive/SI26_Urdu_OCR/data/raw/raw/other/utrset_013.jpg
 - /content/drive/MyDrive/SI26_Urdu_OCR/data/raw/raw/augmented/urdu_043_aug2_rotation.png
 - /content/drive/MyDrive/SI26_Urdu_OCR/data/raw/raw/augmented/urdu_008_aug2_blur.png
 - /content/drive/MyDrive/SI26_Urdu_OCR/data/raw/raw/augmented/urdu_020_aug1_rotation.png


## Section 8 — Build and Launch the Gradio App

**What each part does:**
- `predict_safe()` wraps `predict()` in a `try/except`. If anything goes wrong, the
  error is shown *directly in the output box* instead of failing silently — so you'll
  never again see a blank/placeholder result with no explanation.
- `gr.Interface` auto-builds the upload box, the text output, and the layout.
- `examples=` (if any were found in Section 7) lets you test with one click — pick an
  example, it auto-fills and auto-submits, no need to click Submit yourself.
- `queue()` makes the app handle requests one at a time in order, which avoids race
  conditions if you open multiple tabs.

**Important — read before running:**
This cell runs *indefinitely* (that's what `debug=True` does — it keeps the app alive
so you can use it). That is expected, not an error. Use the printed
`https://....gradio.live` link to open and test the app in a **new browser tab**.
When you're done testing, come back here and click Colab's **Stop** button (■) to end
the cell — do not interrupt it mid-request while a prediction is running, or that one
request will be lost (though the app itself will recover fine on the next run).

**After the link opens:** upload an image (or click one of the examples), then
click **Submit**. The extracted text appears in the box below within a few seconds.
Screenshot this working demo now — you need it for your README (Section 10).

In [ ]:
import gradio as gr
import traceback

def predict_safe(image):
    if image is None:
        return "No image received yet — upload one or pick an example above, then click Submit."
    try:
        result = predict(image)
    except Exception:
        return "ERROR while predicting:\n\n" + traceback.format_exc()

    if not result or not result.strip():
        return "[Model returned empty text for this image — try a clearer or higher-contrast image.]"

    return result


demo = gr.Interface(
    fn=predict_safe,
    inputs=gr.Image(type="pil", label="Upload an image containing Urdu text"),
    outputs=gr.Textbox(label="Extracted Urdu Text"),
    examples=example_image_paths if example_image_paths else None,
    title="Urdu OCR — Code Saviours SI-26",
    description=(
        "Upload a photo or scan containing Urdu text. This tool uses a TrOCR-style "
        "encoder-decoder model, fine-tuned on a custom Urdu dataset, to read the text "
        "and return it as editable Unicode. Pick one of the examples below for an "
        "instant one-click test."
    ),
)

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef51cc1a161e7f18d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Section 9 — Package Everything for Hugging Face Spaces

The `.gradio.live` link from Section 8 is temporary — it expires when this Colab
session ends. Hugging Face Spaces gives you a **permanent** public URL instead.

Before running this cell, create your Space (do this once, on the Hugging Face
website):
1. Go to https://huggingface.co and log in.
2. Click your profile icon → **New Space**.
3. Name it exactly: `urdu-ocr-codesaviours-si26-[yourfirstname]`
4. SDK: **Gradio**. Visibility: **Public**. Click **Create Space**.

This cell then builds a self-contained folder (`spaces_upload/`) with everything the
Space needs — `app.py`, `requirements.txt`, and your model files — with **no
dependency on Google Drive**, since Spaces can't access your Drive at all.

In [ ]:
import shutil

SPACES_DIR = "/content/spaces_upload"
os.makedirs(SPACES_DIR, exist_ok=True)

# 1) Copy the fine-tuned model folder in as-is (weights, image processor config, vocab)
model_dest = os.path.join(SPACES_DIR, "trocr-urdu-finetuned")
if os.path.exists(model_dest):
    shutil.rmtree(model_dest)
shutil.copytree(MODEL_DIR, model_dest)

# 2) requirements.txt — exactly what app.py needs to run on Spaces
requirements_txt = """gradio
torch
transformers
pillow
"""
with open(os.path.join(SPACES_DIR, "requirements.txt"), "w") as f:
    f.write(requirements_txt)

# 3) app.py — the same predict() + Gradio logic as Sections 3-8 above, adapted for Spaces:
#      - no drive.mount (Spaces has no Google Drive access at all)
#      - MODEL_DIR is a relative path, since app.py and trocr-urdu-finetuned/ are committed side by side
app_py = '''import os
import pickle
import traceback

import gradio as gr
import torch
from PIL import Image
from transformers import VisionEncoderDecoderModel, ViTImageProcessor

MODEL_DIR = os.path.join(os.path.dirname(__file__), "trocr-urdu-finetuned")


class UrduCharTokenizerDecode:
    """Decode-only tokenizer rebuilt from the saved Week 4 character vocab."""

    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {i: tok for tok, i in vocab.items()}
        self.pad_token_id = vocab["<pad>"]
        self.bos_token_id = vocab["<bos>"]
        self.eos_token_id = vocab["<eos>"]
        self.unk_token_id = vocab["<unk>"]

    def decode(self, ids, skip_special_tokens=True):
        out = []
        for i in ids:
            i = int(i)
            if skip_special_tokens and i in (self.pad_token_id, self.bos_token_id, self.eos_token_id):
                continue
            out.append(self.inv_vocab.get(i, ""))
        return "".join(out)


with open(os.path.join(MODEL_DIR, "urdu_char_vocab.pkl"), "rb") as f:
    urdu_vocab = pickle.load(f)
tokenizer = UrduCharTokenizerDecode(urdu_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR)
image_processor = ViTImageProcessor.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()


@torch.no_grad()
def predict(image):
    if image is None:
        return ""
    image = image.convert("RGB")
    pixel_values = image_processor(image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


def predict_safe(image):
    if image is None:
        return "Upload an image, then click Submit."
    try:
        result = predict(image)
    except Exception:
        return "ERROR while predicting:\\n\\n" + traceback.format_exc()
    if not result or not result.strip():
        return "[Model returned empty text for this image.]"
    return result


demo = gr.Interface(
    fn=predict_safe,
    inputs=gr.Image(type="pil", label="Upload an image containing Urdu text"),
    outputs=gr.Textbox(label="Extracted Urdu Text"),
    title="Urdu OCR - Code Saviours SI-26",
    description=(
        "Upload a photo or scan containing Urdu text. This tool uses a TrOCR-style "
        "encoder-decoder model, fine-tuned on a custom Urdu dataset, to read the text "
        "and return it as editable Unicode."
    ),
    allow_flagging="never",
)
demo.queue()

if __name__ == "__main__":
    demo.launch()
'''
with open(os.path.join(SPACES_DIR, "app.py"), "w") as f:
    f.write(app_py)

print("Files ready in:", SPACES_DIR)
for item in sorted(os.listdir(SPACES_DIR)):
    print(" -", item)

### Uploading to your Space

**Easiest path (web UI, no terminal needed):**
1. Run the zip cell below, then download `spaces_upload.zip` from the Colab file
   browser (folder icon, left sidebar).
2. Unzip it on your computer.
3. On your Space page: **Files → Add file → Upload files**, then drag in `app.py`,
   `requirements.txt`, and the whole `trocr-urdu-finetuned` folder.
4. Commit. Hugging Face builds the Space automatically — wait 2-3 minutes, then open
   your Space URL. It should be live.

**Alternative (git — faster for large model files):**
```bash
git clone https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname]
# copy the contents of spaces_upload/ into the cloned folder, then:
git add .
git commit -m "Deploy Urdu OCR app"
git push
```
(Spaces automatically uses Git LFS for the model weight files — no extra setup needed.)

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/spaces_upload", "zip", SPACES_DIR)
print("Zipped to:", zip_path)
print("Download it from the Colab file browser (folder icon on the left) and upload its contents to your Space.")

## Section 10 — Generate a README Draft

Builds a starting-point `README.md` covering all 8 required sections from the
handout, pre-filled with the dataset facts from Weeks 1-4.

**You must still edit this before submitting:**
- Replace `[your-username]` and `[yourfirstname]` with your actual Space URL.
- Replace the GitHub URL if your repo name differs.
- Fill in your actual Week 4 accuracy % (from that notebook's Section 12 output).
A README with unfilled brackets in it reads as unfinished — take the extra five
minutes to fill them in.

In [ ]:
readme_md = """# Urdu OCR — A Fine-Tuned TrOCR Model for Extracting Text from Urdu Images

## What problem this solves and why it matters
Optical Character Recognition for Urdu lags far behind Latin-script OCR: Urdu's
cursive, context-dependent Nastaliq script and the scarcity of labeled datasets make
off-the-shelf tools like Tesseract perform poorly out of the box. This project
fine-tunes a TrOCR-style model specifically on Urdu text so it can read real-world
Urdu images — for example, digitizing scanned Urdu documents, signboards, or book
pages that would otherwise have to be transcribed by hand.

## How it works
[TrOCR](https://huggingface.co/microsoft/trocr-base-printed) pairs a vision encoder
(which "looks" at the image) with a text decoder (which "writes out" what it reads),
trained end-to-end on paired image/text data. Since the original TrOCR decoder only
understands English, this project keeps the pretrained visual encoder but replaces
the decoder with a small one built for a custom Urdu character vocabulary, then
fine-tunes it on a labeled Urdu image dataset so it learns to map Urdu glyphs to
Urdu Unicode text.

## Live demo
**[Try it here](https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname])**

## How to run it locally
```bash
git clone https://github.com/qandeelasim13/URDU-OCR-PROJECT-CODE-SAVIOURS-SI-2026-QANDEEL-ASIM.git
cd URDU-OCR-PROJECT-CODE-SAVIOURS-SI-2026-QANDEEL-ASIM
pip install -r requirements.txt
python app.py
```
Then open the local URL Gradio prints in your terminal.

## Dataset details
- 246 labeled Urdu text images from four sources: the UTRSet-Real dataset, synthetic
  images generated with the Noto Nastaliq Urdu font, augmented variants of those
  images, and manual screenshots labeled via EasyOCR.
- Mix of printed book-style and signboard-style text, in varying fonts, backgrounds,
  and image sizes.
- Split 198 train / 50 test.

## Results
Final test-set accuracy: **[fill in from your Week 4 submission output]%**
(measured as 1 minus Character Error Rate).
[If accuracy was lower than expected, briefly explain why — e.g. small dataset size,
limited font variety — and what you'd try with more time, such as more training data
or more training epochs.]

## Credit
Qandeel Asim
Built during the Code Saviours ML/AI Internship — Batch SI-26.
"""

with open(os.path.join(SPACES_DIR, "README.md"), "w") as f:
    f.write(readme_md)

readme_drive_path = os.path.join(PROJECT_DIR, "README_week5_draft.md")
with open(readme_drive_path, "w") as f:
    f.write(readme_md)

print("README draft written to:")
print(" -", os.path.join(SPACES_DIR, "README.md"))
print(" -", readme_drive_path)
print()
print(readme_md)

## Section 11 — Submission Checklist

- [ ] Live Hugging Face Space link (open it fresh and test it before submitting)
- [ ] GitHub repo link with the complete README (all 8 sections filled in, no
      `[brackets]` left unedited)
- [ ] This Week 5 Colab notebook link
- [ ] Screenshot of the working Gradio demo (from Section 8)

Submit all of the above by **Friday, July 31**.

### If something still doesn't work
Run the cells in order top to bottom in a **fresh** Colab runtime
(Runtime → Restart session), without skipping any cell. Almost every "it's not
working" case comes down to either an out-of-order run or the Gradio cell being
interrupted mid-request — both are fixed by a clean top-to-bottom run.